In [1]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
import gc
import os
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

print("--- Step 1: Library Imports and Initial Setup ---")

# --- Global Constants & Paths ---
RANDOM_STATE = 42
N_SPLITS = 5
N_OPTUNA_TRIALS_ERROR = 25 # A good number of trials for the error model
COMPETITION_ALPHA = 0.1
DATA_PATH = './'
KFOLD_PREDS_PATH = './kfold_oof_predictions/'
NN_PREDS_PATH = './NN_model_predictions/'

# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    return np.mean(width + penalty_lower + penalty_upper)

# --- Load Raw Data for Evaluation ---
print("\n--- Step 2: Loading Raw Dataset for Evaluation ---")
try:
    # We only need the original df_train to get the true prices and grade for stratification
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'dataset.csv'))
    y_true = df_train['sale_price'].copy()
    grade_for_stratify = df_train['grade'].copy()
    print("Raw data loaded successfully. 'y_true' and 'grade_for_stratify' are ready.")
except FileNotFoundError as e:
    print(f"ERROR: Could not find 'dataset.csv'. {e}")
    exit()

--- Step 1: Library Imports and Initial Setup ---

--- Step 2: Loading Raw Dataset for Evaluation ---
Raw data loaded successfully. 'y_true' and 'grade_for_stratify' are ready.


In [2]:
# =============================================================================
# BLOCK 2: LOAD DATA, DEFINE & EXECUTE FEATURE ENGINEERING PIPELINE
# =============================================================================
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
import gc

# --- Step 1: Load Raw Data ---
# This step is crucial and is added here to ensure the DataFrames always exist.
print("--- Loading raw data for feature engineering... ---")
DATA_PATH = './'
try:
    drop_cols = ['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm', 'view_otherwater', 'view_other']
    df_train = pd.read_csv(os.path.join(DATA_PATH, 'dataset.csv')).drop(columns=drop_cols)
    df_test = pd.read_csv(os.path.join(DATA_PATH, 'test.csv')).drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError as e:
    print(f"ERROR: Could not find data files. {e}")
    exit()

# Define a random state for reproducibility
RANDOM_STATE = 42

# --- Step 2: Define the Feature Engineering Function ---
def create_comprehensive_features(df_train, df_test):
    """
    Combines original and new advanced feature engineering steps into a single pipeline.
    """
    print("\n--- Starting Comprehensive Feature Engineering ---")

    # Store original indices and target variable
    train_ids = df_train.index
    test_ids = df_test.index
    y_train = df_train['sale_price'].copy() # Keep the target separate

    # Combine for consistent processing
    df_train_temp = df_train.drop(columns=['sale_price'])
    all_data = pd.concat([df_train_temp, df_test], axis=0, ignore_index=True)

    # A) Brute-Force Numerical Interactions
    print("  Step A: Creating numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1', 'grade', 'year_built']
    for col in NUMS:
        if col not in all_data.columns: all_data[col] = 0
        else: all_data[col] = pd.to_numeric(all_data[col], errors='coerce').fillna(0)
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] * all_data[NUMS[j]]

    # B) Date Features
    print("  Step B: Creating date features...")
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['sale_year'] = all_data['sale_date'].dt.year
    all_data['sale_month'] = all_data['sale_date'].dt.month
    all_data['sale_dayofyear'] = all_data['sale_date'].dt.dayofyear
    all_data['age_at_sale'] = all_data['sale_year'] - all_data['year_built']

    # C) TF-IDF Text Features
    print("  Step C: Creating TF-IDF features...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=128, binary=True)
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        all_data = pd.concat([all_data, tfidf_df], axis=1)

    # D) Log transform
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns: all_data[c] = np.log1p(all_data[c].fillna(0))

    # F) Group-By Aggregation Features
    print("  Step F: Creating group-by aggregation features...")
    group_cols = ['submarket', 'city', 'zoning']
    num_cols_for_agg = ['grade', 'sqft', 'imp_val', 'land_val', 'age_at_sale']
    for group_col in group_cols:
        for num_col in num_cols_for_agg:
            agg_stats = all_data.groupby(group_col)[num_col].agg(['mean', 'std', 'max', 'min']).reset_index()
            agg_stats.columns = [group_col] + [f'{group_col}_{num_col}_{stat}' for stat in ['mean', 'std', 'max', 'min']]
            all_data = pd.merge(all_data, agg_stats, on=group_col, how='left')
            all_data[f'{num_col}_minus_{group_col}_mean'] = all_data[num_col] - all_data[f'{group_col}_{num_col}_mean']

    # G) Ratio Features
    print("  Step G: Creating ratio features...")
    epsilon = 1e-6 
    all_data['total_val'] = all_data['imp_val'] + all_data['land_val']
    all_data['imp_val_to_land_val_ratio'] = all_data['imp_val'] / (all_data['land_val'] + epsilon)
    all_data['land_val_ratio'] = all_data['land_val'] / (all_data['total_val'] + epsilon)
    all_data['sqft_to_lot_ratio'] = all_data['sqft'] / (all_data['sqft_lot'] + epsilon)
    all_data['was_renovated'] = (all_data['year_reno'] > 0).astype(int)
    all_data['reno_age_at_sale'] = np.where(all_data['was_renovated'] == 1, all_data['sale_year'] - all_data['year_reno'], -1)

    # H) Geospatial Clustering Features
    print("  Step H: Creating geospatial clustering features...")
    coords = all_data[['latitude', 'longitude']].copy().fillna(all_data[['latitude', 'longitude']].median())
    kmeans = KMeans(n_clusters=20, random_state=RANDOM_STATE, n_init=10) 
    all_data['location_cluster'] = kmeans.fit_predict(coords)
    cluster_centers = kmeans.cluster_centers_
    for i in range(len(cluster_centers)):
        center = cluster_centers[i]
        all_data[f'dist_to_cluster_{i}'] = np.sqrt((coords['latitude'] - center[0])**2 + (coords['longitude'] - center[1])**2)

    # --- Final Cleanup ---
    print("  Step I: Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city', 'sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data = pd.get_dummies(all_data, columns=['location_cluster'], prefix='loc_cluster')
    object_cols = all_data.select_dtypes(include='object').columns
    if len(object_cols) > 0: all_data = all_data.drop(columns=object_cols)
    all_data.fillna(0, inplace=True)
    
    # Separate back into train and test sets
    train_len = len(train_ids)
    X = all_data.iloc[:train_len].copy()
    X_test = all_data.iloc[train_len:].copy()
    X.index, X_test.index = train_ids, test_ids
    X_test = X_test[X.columns]
    
    print(f"\nComprehensive FE complete. Total features: {X.shape[1]}")
    gc.collect()
    
    return X, X_test, y_train

# --- Step 3: Execute the Feature Engineering Pipeline ---
print("\n--- Executing the full feature engineering pipeline... ---")
X, X_test, y_train = create_comprehensive_features(df_train, df_test)

# --- Step 4: Verify the Output ---
print(f"\nFeature engineering complete. Final shapes are:")
print(f"  X (train features): {X.shape}")
print(f"  X_test (test features): {X_test.shape}")
print(f"  y_train (target): {y_train.shape}")
gc.collect()

--- Loading raw data for feature engineering... ---
Raw data loaded successfully.

--- Executing the full feature engineering pipeline... ---

--- Starting Comprehensive Feature Engineering ---
  Step A: Creating numerical interaction features...
  Step B: Creating date features...
  Step C: Creating TF-IDF features...
  Step F: Creating group-by aggregation features...
  Step G: Creating ratio features...
  Step H: Creating geospatial clustering features...
  Step I: Finalizing feature set...

Comprehensive FE complete. Total features: 233

Feature engineering complete. Final shapes are:
  X (train features): (200000, 233)
  X_test (test features): (200000, 233)
  y_train (target): (200000,)


0

In [3]:
# =============================================================================
# BLOCK 3: LOAD & VALIDATE PRE-COMPUTED PREDICTIONS
# =============================================================================
print("\n--- Starting Block 3: Loading All Pre-computed Predictions ---")

try:
    # Load XGBoost, CatBoost, and Neural Network predictions
    oof_xgb_preds = np.load(os.path.join(KFOLD_PREDS_PATH, 'oof_xgb_preds.npy'))
    test_xgb_preds = np.load(os.path.join(KFOLD_PREDS_PATH, 'test_xgb_preds.npy'))
    oof_catboost_preds = np.load(os.path.join(KFOLD_PREDS_PATH, 'oof_catboost_preds.npy'))
    test_catboost_preds = np.load(os.path.join(KFOLD_PREDS_PATH, 'test_catboost_preds.npy'))
    oof_nn_preds = np.load(os.path.join(NN_PREDS_PATH, 'oof_nn_preds.npy'))
    test_nn_preds = np.load(os.path.join(NN_PREDS_PATH, 'test_nn_preds.npy'))
    print("All .npy prediction files loaded successfully.")
except FileNotFoundError as e:
    print(f"ERROR: A required .npy prediction file was not found. {e}")
    print("Please ensure all training notebooks have been run successfully.")
    exit()

# --- FORENSIC ANALYSIS: Check All Loaded Prediction Arrays ---
print("\n--- FORENSIC ANALYSIS of all loaded predictions ---")
all_preds_clean = True
for name, arr in [("OOF XGB", oof_xgb_preds), ("Test XGB", test_xgb_preds), 
                  ("OOF CB", oof_catboost_preds), ("Test CB", test_catboost_preds),
                  ("OOF NN", oof_nn_preds), ("Test NN", test_nn_preds)]:
    
    print(f"\nDescribing '{name}' array:")
    print(pd.Series(arr).describe())
    
    if np.isnan(arr).any() or np.isinf(arr).any() or np.min(arr) < -10000:
        print(f"*** CRITICAL WARNING: Bad values found in '{name}' array! ***")
        all_preds_clean = False

if all_preds_clean:
    print("\nSUCCESS: All loaded prediction arrays are clean and appear valid.")
else:
    print("\nERROR: Corrupted data detected. Please re-run the appropriate training notebook.")
    exit()


--- Starting Block 3: Loading All Pre-computed Predictions ---
All .npy prediction files loaded successfully.

--- FORENSIC ANALYSIS of all loaded predictions ---

Describing 'OOF XGB' array:
count    2.000000e+05
mean     5.843142e+05
std      4.038169e+05
min      3.466805e+04
25%      3.095954e+05
50%      4.643079e+05
75%      7.217962e+05
max      2.948615e+06
dtype: float64

Describing 'Test XGB' array:
count    2.000000e+05
mean     5.928046e+05
std      4.097348e+05
min      4.257055e+04
25%      3.132094e+05
50%      4.717674e+05
75%      7.324603e+05
max      2.925905e+06
dtype: float64

Describing 'OOF CB' array:
count    2.000000e+05
mean     5.841072e+05
std      4.048227e+05
min      2.986686e+03
25%      3.093048e+05
50%      4.637185e+05
75%      7.218511e+05
max      3.298709e+06
dtype: float64

Describing 'Test CB' array:
count    2.000000e+05
mean     5.922556e+05
std      4.098405e+05
min     -1.708337e+03
25%      3.128933e+05
50%      4.716087e+05
75%      7.3141

In [ ]:
# =============================================================================
# BLOCK 4 & 5 (REVISED): OPTIMIZED ENSEMBLE, ERROR MODEL TUNING & SAVING
# =============================================================================
print("\n--- Starting Block 4: Optimizing 3-Model Ensemble Weights ---")

# (The first part of optimizing the mean ensemble weights remains the same)
def get_ensemble_rmse(w): return np.sqrt(mean_squared_error(y_true, w[0]*oof_xgb_preds + w[1]*oof_catboost_preds + w[2]*oof_nn_preds))
res = minimize(get_ensemble_rmse, [1/3]*3, method='SLSQP', bounds=[(0,1)]*3, constraints={'type':'eq','fun':lambda w:1-np.sum(w)})
best_weights = res.x
print(f"Optimal Weights -> XGB: {best_weights[0]:.3f}, CB: {best_weights[1]:.3f}, NN: {best_weights[2]:.3f}")
oof_ensemble_mean = best_weights[0]*oof_xgb_preds + best_weights[1]*oof_catboost_preds + best_weights[2]*oof_nn_preds
test_ensemble_mean = best_weights[0]*test_xgb_preds + best_weights[1]*test_catboost_preds + best_weights[2]*test_nn_preds
print(f"\nOptimized Ensemble OOF RMSE: ${np.sqrt(mean_squared_error(y_true, oof_ensemble_mean)):,.2f}")

# --- Starting the Error Model Pipeline ---
print("\n" + "="*80)
print("--- Starting Block 5: Tuning and Training the Ensemble Error Model ---")
print("="*80)

# --- A. Define the Error Target and Feature Set ---
error_target = np.abs(y_true - oof_ensemble_mean)
X_for_error = X.copy()
X_for_error['mean_ensemble_pred'] = oof_ensemble_mean
print("Error target and feature set created for training.")

# --- B. Tune the Error Model with Optuna ---
print("\n--- Step B: Tuning the XGBoost Error Model with Optuna ---")
def objective_error_model(trial):
    # Use a single train/validation split for faster tuning
    X_train_err_opt, X_val_err_opt, y_train_err_opt, y_val_err_opt = train_test_split(
        X_for_error, error_target, test_size=0.2, random_state=RANDOM_STATE
    )
    dtrain = xgb.DMatrix(X_train_err_opt, label=y_train_err_opt)
    dval = xgb.DMatrix(X_val_err_opt, label=y_val_err_opt)
    
    # Define a search space optimized for error modeling (often simpler models work best)
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'tree_method': 'hist', 'n_jobs': -1, 'seed': RANDOM_STATE,
        'eta': trial.suggest_float('eta', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'lambda': trial.suggest_float('lambda', 1e-2, 20.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-2, 20.0, log=True)
    }
    
    bst = xgb.train(params, dtrain, num_boost_round=2000, evals=[(dval, 'val')], early_stopping_rounds=75, verbose_eval=False)
    preds = bst.predict(dval, iteration_range=(0, bst.best_iteration))
    return np.sqrt(mean_squared_error(y_val_err_opt, preds))

study_error = optuna.create_study(direction='minimize')
study_error.optimize(objective_error_model, n_trials=30)
best_params_error = study_error.best_params
print("Error model tuning complete. Best validation RMSE: ${study_error.best_value:,.2f}")
print("Best hyperparameters for error model:", best_params_error)

# --- C. K-Fold Train and Save the Tuned Error Model ---
print("\n--- Step C: K-Fold training and saving the final error models ---")
ERROR_MODELS_PATH = './error_models/'
os.makedirs(ERROR_MODELS_PATH, exist_ok=True)
print(f"Trained error models will be saved in: '{ERROR_MODELS_PATH}'")

oof_error_preds, test_error_preds = np.zeros(len(X)), np.zeros(len(X_test))
X_test_for_error = X_test.copy()
X_test_for_error['mean_ensemble_pred'] = test_ensemble_mean
X_test_for_error = X_test_for_error[X_for_error.columns] # CRITICAL FIX

skf_err = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (train_idx, val_idx) in enumerate(skf_err.split(X_for_error, grade_for_stratify)):
    print(f"  Training and saving error model for fold {fold+1}/{N_SPLITS}...")
    dtrain_err = xgb.DMatrix(X_for_error.iloc[train_idx], label=error_target.iloc[train_idx])
    dval_err = xgb.DMatrix(X_for_error.iloc[val_idx], label=error_target.iloc[val_idx])
    
    # Train the model for this fold
    bst_error = xgb.train(
        best_params_error,
        dtrain_err,
        num_boost_round=3000,
        evals=[(dval_err, 'v')],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    
    # Save the trained model for this specific fold
    model_filename = os.path.join(ERROR_MODELS_PATH, f'xgb_error_model_fold_{fold+1}.json')
    bst_error.save_model(model_filename)
    
    # Generate OOF and Test predictions
    oof_error_preds[val_idx] = bst_error.predict(xgb.DMatrix(X_for_error.iloc[val_idx]), iteration_range=(0, bst_error.best_iteration))
    test_error_preds += bst_error.predict(xgb.DMatrix(X_test_for_error), iteration_range=(0, bst_error.best_iteration)) / N_SPLITS

print(f"\nFinal Error Model OOF RMSE: ${np.sqrt(mean_squared_error(error_target, oof_error_preds)):,.2f}")
print("All error models trained and saved successfully.")

[I 2025-07-23 00:31:19,786] A new study created in memory with name: no-name-4830dadf-b5da-4dcf-814c-3531d60e302c



--- Starting Block 4: Optimizing 3-Model Ensemble Weights ---
Optimal Weights -> XGB: 0.437, CB: 0.476, NN: 0.087

Optimized Ensemble OOF RMSE: $95,201.98

--- Starting Block 5: Tuning and Training the Ensemble Error Model ---
Error target and feature set created for training.

--- Step B: Tuning the XGBoost Error Model with Optuna ---


[I 2025-07-23 00:31:46,388] Trial 0 finished with value: 61309.29222954992 and parameters: {'eta': 0.011777825902856828, 'max_depth': 7, 'subsample': 0.6556905754069928, 'colsample_bytree': 0.5936880516851286, 'lambda': 0.7124399670561512, 'alpha': 16.223651102694507}. Best is trial 0 with value: 61309.29222954992.
[I 2025-07-23 00:31:54,862] Trial 1 finished with value: 61871.726981567 and parameters: {'eta': 0.03762213318507882, 'max_depth': 5, 'subsample': 0.5784812609596233, 'colsample_bytree': 0.5201988082371997, 'lambda': 0.01656006779711972, 'alpha': 0.10497002156038439}. Best is trial 0 with value: 61309.29222954992.
[I 2025-07-23 00:32:03,938] Trial 2 finished with value: 61702.27611772605 and parameters: {'eta': 0.032831309994626, 'max_depth': 5, 'subsample': 0.7612771426519858, 'colsample_bytree': 0.6326242310161375, 'lambda': 0.42168738294507707, 'alpha': 1.6168114926954311}. Best is trial 0 with value: 61309.29222954992.


In [ ]:
# =============================================================================
# BLOCK 6: FINAL CALIBRATION AND SUBMISSION
# =============================================================================
print("\n--- Starting Block 6: Final Analysis, Calibration, and Submission ---")

oof_error_final, test_error_final = np.clip(oof_error_preds, 0, None), np.clip(test_error_preds, 0, None)

print("\n--- FINAL FORENSIC ANALYSIS ---")
print("Describing 'test_ensemble_mean':\n", pd.Series(test_ensemble_mean).describe())
print("\nDescribing 'test_error_final':\n", pd.Series(test_error_final).describe())

print("\n--- Calibrating intervals with Winkler Score... ---")
best_score, best_a, best_b = float('inf'), 1.0, 1.0
for a in np.arange(1.8, 2.8, 0.01):
    for b in np.arange(1.8, 2.8, 0.01):
        score = winkler_score(y_true, oof_ensemble_mean - oof_error_final * a, oof_ensemble_mean + oof_error_final * b)
        if score < best_score: best_score, best_a, best_b = score, a, b

print("\n--- Creating and Verifying Submission File ---")
final_lower = test_ensemble_mean - test_error_final * best_a
final_upper = test_ensemble_mean + test_error_final * b
final_upper = np.maximum(final_lower, final_upper)
submission_df = pd.DataFrame({'id': pd.read_csv(os.path.join(DATA_PATH,'test.csv'))['id'], 'pi_lower': final_lower, 'pi_upper': final_upper})

if submission_df.isnull().sum().any(): print("\n\n*** CRITICAL WARNING: Null values detected in submission! ***\n")
else: print("\nSUCCESS: Final submission DataFrame is clean and ready.")

print("\n" + "="*60)
print("--- FINAL RESULTS & SUBMISSION ---")
print("="*60)
print(f"Best OOF Winkler Score: {best_score:,.2f}")
print(f"Optimal Multipliers: a={best_a:.3f}, b={best_b:.3f}")

submission_filename = f'submission_final_3model_winkler_{int(best_score)}.csv'
submission_df.to_csv(submission_filename, index=False)
print(f"\n'{submission_filename}' created successfully! Good luck on the leaderboard!")
print("\nSubmission Head:\n", submission_df.head())